# 家庭作业练习作业

升级第 1 天项目以总结网页，以使用通过 Ollama 而不是 OpenAI 在本地运行的开源模型

为使用 JavaScript 加载内容的网站添加对 Selenium 抓取的支持。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import os
import requests
from scraper import fetch_website_contents, fetch_website_contents_selenium
from IPython.display import Markdown, display
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434"

In [ ]:
# 检查 Ollama 是否正在运行
requests.get(f'{OLLAMA_BASE_URL}').content

### 从元下载 llama3.2

如果您的计算机较小，请将其更改为 llama3.2:1b。

不要使用 llama3.3 或 llama4！它们对你的电脑来说太大了..

In [ ]:
# 下载 llama3.2 到您的机器上
!ollama pull llama3.2

In [ ]:
# 使用 Ollama 创建 OpenAI 客户端
ollama = OpenAI(base_url=f'{OLLAMA_BASE_URL}/v1', api_key='ollama')

**可选（对于 Selenium scraper）：** 从终端运行：`uv pip install selenium webdriver-manager`

重新启动 Jupyter 内核，以便它获取新软件包。

这可以避免从笔记本内部运行 uv 或 pip。

In [ ]:
# 定义 WebsiteSummarizer 类
class WebsiteSummarizer:
    client = ollama
    system_prompt =  """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
    user_prompt_prefix =  """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.  
"""

    def __init__(self, url, use_selenium=False):
        self.url = url
        # 对于使用 JavaScript 加载内容的网站，请使用 Selenium 方法
        fetcher = fetch_website_contents_selenium if use_selenium else fetch_website_contents
        self.contents = fetcher(url)
        self.summary = self.summarize()
        return self.display_summary()

    def messages_for(self):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f'{self.user_prompt_prefix}\n\n{self.contents}'}
        ]

    def summarize(self):
        response = self.client.chat.completions.create(
            model="llama3.2",
            messages=self.messages_for()
        )
        return response.choices[0].message.content
      
    def display_summary(self):
        display(Markdown(self.summary))   

In [ ]:
# 演示
WebsiteSummarizer("https://www.andela.com")

In [ ]:
# 演示 - 使用 JavaScript 加载内容的站点
WebsiteSummarizer("https://dheerajmaddi.netlify.app/", use_selenium=True)